# META-CXR — Table 5 encoder sensitivity

This notebook loads the single validation-selected E123 checkpoint, runs BioViL-T + PubMedCLIP + SwinV2 once per held-out batch, evaluates the six encoder subsets reported in paper Table 5, and displays only the final table. Inactive token spans are removed before MHCAC. No Kaggle results dataset is created or uploaded.

In [ ]:
DATASET_SLUG = "phuong20052/mimic-cxr-jpg-dataset"
CHECKPOINT_GCS_BUCKET = "meta-cxr-checkpoints-phuongnm"  # bucket holding checkpoint_best.pth at its root
REPO_REF = "main"  # resolve and record the exact evaluation commit after fetch
BATCH_SIZE = 2
NUM_WORKERS = 4
SEED = 42


In [ ]:
import os, pathlib, subprocess, sys
for name, value in {'DATASET_SLUG': DATASET_SLUG, 'CHECKPOINT_GCS_BUCKET': CHECKPOINT_GCS_BUCKET, 'REPO_REF': REPO_REF}.items():
    if not value:
        raise ValueError(f'{name} is required')
repo_dir = pathlib.Path('/kaggle/working/META-CXR-SMOKETEST')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', '--no-checkout', 'https://github.com/minhphuong150505/META-CXR-SMOKETEST.git', str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'fetch', '--depth=1', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
if len(REPO_COMMIT) != 40:
    raise RuntimeError('Could not resolve an exact evaluation commit')
print('Evaluation source commit:', REPO_COMMIT)
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
os.environ['PYTHONPATH'] = str(repo_dir) + os.pathsep + os.environ.get('PYTHONPATH', '')


In [ ]:
# Checkpoints contain non-weight provenance and RNG state.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'


In [ ]:
# Only the private GCS checkpoint credential is needed for evaluation.
from smoke.runtime import load_kaggle_secrets
load_kaggle_secrets(('GCS_SERVICE_ACCOUNT',), '/tmp/.meta-cxr-secrets')
print('Loaded the GCS checkpoint credential (value hidden).')


In [ ]:
import json
from smoke.runtime import environment_fingerprint, assert_two_t4, compatibility_matrix
before = environment_fingerprint()
print(json.dumps(before, indent=2, sort_keys=True))
assert_two_t4(before)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', 'requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', 'hi-ml-multimodal==0.2.1'], check=True)
after = environment_fingerprint()
assert_two_t4(after)
print(json.dumps(compatibility_matrix(before, after), indent=2, sort_keys=True))


In [ ]:
from smoke.runtime import discover_dataset, load_dataset_manifest, write_runtime_env_config
from smoke.checkpoints import download_best_checkpoint
dataset_root = discover_dataset(DATASET_SLUG)
dataset_manifest, _, dataset_hash = load_dataset_manifest(dataset_root)
if dataset_manifest.get('status') != 'qa_passed':
    raise RuntimeError('Dataset manifest is not QA-passed')
# Keep the checkpoint and evaluator artifacts outside /kaggle/working so they
# are not published as notebook output.
runtime_dir = pathlib.Path('/tmp/meta-cxr-table5')
write_runtime_env_config(dataset_root, runtime_dir)
checkpoint = download_best_checkpoint(CHECKPOINT_GCS_BUCKET, runtime_dir / 'checkpoint')
print('Downloaded validation-selected checkpoint to temporary storage:', checkpoint)


In [ ]:
import torch
from IPython.display import Markdown, display
checkpoint_meta = torch.load(checkpoint, map_location='cpu')
identity = checkpoint_meta.get('identity')
if not identity or identity.get('dataset_manifest_sha256') != dataset_hash:
    raise RuntimeError('Checkpoint dataset identity mismatch')
result_path = pathlib.Path('/tmp/meta-cxr-table5/encoder_sensitivity.json')
eval_env = os.environ.copy()
eval_env['PYTHONPATH'] = str(repo_dir) + os.pathsep + eval_env.get('PYTHONPATH', '')
cmd = [sys.executable, 'scripts/evaluate_encoder_sensitivity.py', '--cfg-path', 'pretraining/configs/stage1_smoke_2xt4.yaml', '--checkpoint', str(checkpoint), '--output', str(result_path), '--overwrite', '--source-commit', REPO_COMMIT, '--dataset-manifest-sha256', dataset_hash, '--config-fingerprint', identity['config_fingerprint'], '--batch-size', str(BATCH_SIZE), '--num-workers', str(NUM_WORKERS)]
completed = subprocess.run(cmd, check=False, env=eval_env, text=True, capture_output=True)
if completed.returncode != 0:
    raise RuntimeError('Table 5 evaluation failed:\n' + completed.stdout[-4000:] + completed.stderr[-4000:])
summary = json.loads(result_path.read_text())
paper_rows = (
    ('E1', '✓', '−', '−'),
    ('E2', '−', '✓', '−'),
    ('E3', '−', '−', '✓'),
    ('E12', '✓', '✓', '−'),
    ('E13', '✓', '−', '✓'),
    ('E123', '✓', '✓', '✓'),
)
lines = [
    '### TABLE 5. Mean F1 score across 5 common abnormalities',
    '',
    '| RN50 | ViT | Swin | Mean F1 Score |',
    '|:---:|:---:|:---:|---:|',
]
for run_id, rn50, vit, swin in paper_rows:
    score = summary['reports'][run_id]['paper_table5']['mean_weighted_f1']
    lines.append(f'| {rn50} | {vit} | {swin} | {score:.3f} |')
display(Markdown('\n'.join(lines)))
